In [24]:
%run board.ipynb

### Monte Carlo Tree Search (MCTS)

MCTS é um algoritmo de busca heurística popular para jogos adversariais pois ele não precisa aprender as regras do jogo para jogar bem, ele aprende através da experiência de milhões de simulações para cada uma de suas jogadas.

O funcionamento de MCTS corresponde a 4 fases (Seleção, Expensão, Simulação, Retropropagação) que serão exploradas a medida em que o código for construído.

#### Base (Nó)

Como o nome do algoritmo sugere, uma árvore de busca é uma estrutura composta por "nós" que representam estados do jogo e outras informações, aqui abaixo uma lista desses informações em forma de atributos da nossa clase:

- board = Estado atual do tabuleiro e definição do jogador da vez.
- parent = Referência ao nó pai, permitindo que os resultados das simulações subam na árvore (retropropagação).
- children = Lista de nós sucessores que já foram expandidos e anexados a este estado.
- move = A ação específica que transformou o estado do pai no estado atual.
- wins = Pontuação acumulada das simulações que passaram por este nó, onde vitórias somam 1.0 e empates 0.5.
- visits = Contador de quantas vezes este nó foi selecionado ou atravessado durante as iterações do algoritmo.
- untried_moves = Lista de movimentos legais gerados pelo tabuleiro que ainda não foram explorados para criar nós filhos.
- player_who_just_moved = O jogador ('X' ou 'O') que executou o movimento para chegar a este nó.

In [25]:
import math
import random

class Node:
    def __init__(self, board, parent=None, move=None):
        self.board = board
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0.0
        self.visits = 0
        self.untried_moves = board.get_valid_moves()
        self.player_who_just_moved = parent.board.current_player if parent else None

#### Métodos

O algoritmo MCTS utiliza de UpperConfidenceBound(UCB1) para tomada de decisões, o UCB1 combina a explotação (média de vitórias atuais) com a exploração (um bônus para nós menos visitados), garantindo que o sistema teste novos caminhos enquanto foca nos mais promissores. Se um nó nunca foi visitado, ele retorna um valor infinito.

o valor da "constante" c, representa o grau de otimismo do nosso algoritmo, para um c alto, nosso algoritmo explora muitos nós em niveis superficiais antes de se aprofundar em uma ramificação. Para um c baixo, nosso algoritmo foca em possibilidades superficialmente boas mas que a longo prazo podem não ser as melhores.

O valor "constante" para c de 1,414, é na verdade aproximação de raiz quadrada de 2, número teórico ideal para encontrar a melhor possibilidade com o menor número de erros possíveis.

Esse cálculo é realizado pelo método *ucb1*.

E sua aplicação em todos os nós filhos do nó atual, é feito pelo método *best_child*, fundamental na etapa de Seleção.

In [26]:
def ucb1(self, c=1.414):
    if self.visits == 0:
        return float('inf')
    exploitation = self.wins / self.visits
    exploration = c * math.sqrt(math.log(self.parent.visits) / self.visits)
    return exploitation + exploration

def best_child(self, c=1.414):
    return max(self.children, key=lambda n: n.ucb1(c))

# Monkey Patching #
Node.ucb1 = ucb1
Node.best_child = best_child

O método *is_fully_expanded* retorna **TRUE** se não houverem mais movimentos possíveis, não explorados, a partir de um determinado nó. Essencial para que nosso algoritmo se aprofunde em profundidade em nossa árvore.

Já o método *is_terminal* é retorna **TRUE** caso o nó atual seja representativo de um estado terminal, sinalizando que não há mais a necessidade de explorar jogadas a partir desse nó.

In [27]:
def is_fully_expanded(self):
    return len(self.untried_moves) == 0

def is_terminal(self):
    return (self.board.check_win('X') or 
            self.board.check_win('O') or 
            not self.board.get_valid_moves())

# Monkey Patching #
Node.is_fully_expanded = is_fully_expanded
Node.is_terminal = is_terminal

O método abaixo representa a etapa de **SIMULAÇÃO** do algoritmo MCTS. Esse etapa consiste na simulação várias jogadas aleatórias a partir de um nó inicial com o objetivo de identificar se o jogo pode resultar em uma vitória, derrota ou empate para determinar a qualidade de um nó.

A simulação é feita através de uma cópia do tabuleiro e possui profundidade máxima determinada de 60, evitando fluxos infinitos ou empates por repetição.

Se o resultado for vitória, a função retorna 1. Se o resultado for derrota, a função retorna 0. Se o resultado for empate, a função retorna 0.5.



In [28]:
def _rollout_vanilla(board, ai_piece, max_depth=60):
    sim_board = board.copy()

    for _ in range(max_depth):
        legal_moves = sim_board.get_valid_moves()
        if not legal_moves:
            return 0.5 

        move = random.choice(legal_moves)
        current_p = sim_board.current_player
        opponent_p = 'O' if current_p == 'X' else 'X'
        sim_board.apply_move(move)

        if move[0] == 'pop':
            cw = sim_board.check_win(current_p)
            ow = sim_board.check_win(opponent_p)
            if cw and ow: return 1.0 if current_p == ai_piece else 0.0
            elif cw: return 1.0 if current_p == ai_piece else 0.0
            elif ow: return 0.0 if current_p == ai_piece else 1.0
        else:
            if sim_board.check_win(current_p):
                return 1.0 if current_p == ai_piece else 0.0
    return 0.5 

# Monkey Patching #
Node._rollout_vanilla = _rollout_vanilla

#### Execução

O método abaixo é responsável por unir os métodos anteriores na sequência lógica do algoritmo de MCTS: SELEÇÃO -> EXPENSÃO -> SIMULAÇÃO -> RETROPROPAGAÇÃO.

Ele utiliza do parâmetro c = 1.414 como informado anteriormente e do número de iterações padronizado em 10000 conforme orientações obtidas em sala de aula, ou seja, o algoritmo irá chegar a um resultado final de 10000 jogos para decidir qual deverá ser o melhor movimento a ser realizado.

Na iteração nº1, por exemplo:

Como nossa raiz não tem todos os filhos explorados, um movimento é selecionado aleatoriamente. E a partir dessa seleção é criado um objeto "nó" associado a raiz, esse nó irá passar pela etapa de simulação, onde retornará o valor de 1, 0, ou 0.5 através da backpropagation até a raiz. As demais iterações serão realizadas dessa forma até que nossa raiz não tenha mais jogadas disponíveis para serem feitas.

Na iteração posterior até todos os filhos da raiz serem explorados, a seleção será feita através do cálculo do UCB1 de todos os filhos, aquele que tiver maior valor será selecionado e a partir dele uma nova expansão será feita. Uma jogada aleatória será selecionada e passará pela etapa de simulação + backpropagation, resultando na alteração do valor de UCB1 dos nós envolvidos na backpropagation.

A partir desse ponto, como na iteração anterior, nosso algoritmo sempre irá iniciar calculando o UCB1 dos nós filhos da raiz, e o UCB1 dos filhos do nó selecionado a partir da raiz, e assim em diante até alcançar o número de 2000 simulações.

Ao fim, o nó com mais visitas representará o movimento escolhido pelo algoritmo.

Comentários sobre a construção do código está no trecho abaixo:

In [29]:
def mcts_vanilla_best_move(board, iterations=10000, c=1.414):
    ai_piece = board.current_player
    root = Node(board.copy())

    for _ in range(iterations):
        node = root
        '''Etapa de seleção, enquanto o nó atual já tiver todos os seus
        filhos mapeados (totalmente expandido), o algoritmo utiliza o
        critério UCB1 para decidir qual ramo da árvore deve aprofundar'''
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child(c)

        '''Etapa de expansão, ao chegar na "fronteira" do conhecimento, 
        escolhemos um movimento inédito para criar um novo nó filho.'''
        if not node.is_terminal() and node.untried_moves:
            move = random.choice(node.untried_moves)
            node.untried_moves.remove(move)
            new_board = node.board.copy()
            new_board.apply_move(move)
            child = Node(new_board, parent=node, move=move)
            node.children.append(child)
            node = child

        '''Etapa de simulação, onde retornará 1.0, 0, ou 0.5'''
        result = _rollout_vanilla(node.board, ai_piece)

        '''Etapa de retropropagação, onde atualizamos as
        estatísticas de vitória e visitação.'''
        backprop_node = node
        while backprop_node is not None:
            backprop_node.visits += 1
            if backprop_node.player_who_just_moved == ai_piece:
                backprop_node.wins += result
            elif backprop_node.player_who_just_moved is not None:
                backprop_node.wins += (1.0 - result)
            backprop_node = backprop_node.parent

    if not root.children: return board.get_valid_moves()[0]
    return max(root.children, key=lambda n: n.visits).move

#### Avaliação do valor de C

Antes de prosseguir com métodos heurísticos que pudessem afetar a execução do algoritmo MCTS, a equipe optou por refinar seus hiperparâmetros, realizando um ajuste fino (tuning) adequado à natureza do jogo PopOut. A metodologia aplicada foi estruturada em três fases complementares: primeiro, realiza-se uma busca em grade abrangente (coarse grid search) para mapear o espaço de busca e identificar regiões promissoras do parâmetro de exploração ($C$) da fórmula UCB1 por meio de simulações estatísticas em arenas de torneio.

A função *simulate_match* foi copiada do notebook arena.ipynb, considerando a necessidade de simulações para determinação de um melhor valor de C.

In [30]:
def simulate_match(p1_name, p1_func, p2_name, p2_func):
    board = Board()
    state_history = {board.get_state(): 1}
    
    while True:
        piece = board.current_player
        opponent_piece = 'O' if piece == 'X' else 'X'
        current_func = p1_func if piece == 'X' else p2_func
        current_name = p1_name if piece == 'X' else p2_name
        opponent_name = p2_name if piece == 'X' else p1_name
        current_state = board.get_state()
        if state_history.get(current_state, 0) >= 3:
            return "Draw" 
        move = current_func(board)
        col = move[1]
        if move[0] == "pop":
            board.pop_piece(col)
            cw = board.check_win(piece)
            ow = board.check_win(opponent_piece)
            if cw and ow: return current_name 
            elif cw: return current_name
            elif ow: return opponent_name
        elif move[0] == "push":
            board.drop_piece(col, piece)
            if board.check_win(piece): return current_name
        new_state = board.get_state()
        state_history[new_state] = state_history.get(new_state, 0) + 1
        if not board.get_valid_moves():
            return "Draw"  
        board.switch_player()

Já a função *play_a_match* foi implementada para obter o resultado entre a disputa entre 2 MCTS com valores do hiperparâmetro C diferentes.

In [31]:
def play_a_match(args):
    nome_c1, c1, nome_c2, c2, iterations, c1_e_X = args
    f_c1 = lambda b: mcts_vanilla_best_move(b, iterations=iterations, c=c1)
    f_c2 = lambda b: mcts_vanilla_best_move(b, iterations=iterations, c=c2)
    if c1_e_X:
        vencedor = simulate_match(nome_c1, f_c1, nome_c2, f_c2)
    else:
        vencedor = simulate_match(nome_c2, f_c2, nome_c1, f_c1)
    if vencedor == nome_c1:
        return 'c1'
    elif vencedor == nome_c2:
        return 'c2'
    else:
        return 'empate'

A função evaluate_c_matchup é a base da avaliação empírica. Ela estrutura os torneios entre duas instâncias do algoritmo MCTS que diferem apenas no valor de $C$. A execução ocorre de forma puramente sequencial, submetendo cada configuração ao mesmo número de partidas como primeiro e segundo jogador, de modo a anular a vantagem de iniciativa. O vencedor é determinado por quem obtém o maior número de vitórias; em caso de empate estatístico, privilegia-se o valor mais próximo da raiz de 2 ($1.414$), que é o padrão teórico da fórmula UCB1.

In [32]:
def evaluate_c_matchup(c1, c2, num_games_per_side=50, iterations=10000):
    name_c1 = f"MCTS_C_{c1:.4f}"
    name_c2 = f"MCTS_C_{c2:.4f}"
    tasks = []
    
    for _ in range(num_games_per_side):
        tasks.append((name_c1, c1, name_c2, c2, iterations, True))
    for _ in range(num_games_per_side):
        tasks.append((name_c1, c1, name_c2, c2, iterations, False))
        
    print(f"  -> {len(tasks)} jogos no total (execução sequencial)...")
    
    results = []
    for task in tasks:
        results.append(play_a_match(task)) 
        
    c1_wins = sum(1 for r in results if r == 'c1')
    c2_wins = sum(1 for r in results if r == 'c2')
    draws   = sum(1 for r in results if r == 'empate')
    
    print(f"  Resultados: {name_c1}: {c1_wins} | {name_c2}: {c2_wins} | Empates: {draws}")
    
    if c2_wins != c1_wins:
        return c2_wins > c1_wins
    else:
        return abs(c2 - 1.414) <= abs(c1 - 1.414)

A função optimize_c_binary_search aplica a heurística de busca binária para afunilar iterativamente o intervalo ótimo de exploração. Em cada iteração (step), calcula o centroide do espaço de busca atual e gera dois valores concorrentes extremamente próximos a este ponto. A instância vencedora do confronto dita se a metade inferior ou superior do intervalo deve ser descartada, promovendo uma convergência rápida para um ótimo local.

In [33]:
def optimize_c_binary_search(low=1.0, high=2.0, max_steps=5, iterations=10000, num_games_per_side=50):
    print(f"====== INICIANDO BUSCA DO VALOR IDEAL DE C NO INTERVALO [{low}, {high}] ======")
    print(f"       {num_games_per_side * 2} jogos/passo | {iterations} iterações\n")
    
    for step in range(1, max_steps + 1):
        mid = (low + high) / 2
        delta = (high - low) * 0.1
        c1 = mid - delta
        c2 = mid + delta
        
        print(f"Passo {step}/{max_steps}: Intervalo atual [{low:.4f}, {high:.4f}]")
        print(f"Testando C1 = {c1:.4f} vs C2 = {c2:.4f}")
        
        c2_won = evaluate_c_matchup(c1, c2, num_games_per_side=num_games_per_side, iterations=iterations)
        
        if c2_won:
            print(f"👉 C2 venceu. Ajustando limite inferior para {mid:.4f}\n")
            low = mid
        else:
            print(f"👉 C1 venceu. Ajustando limite superior para {mid:.4f}\n")
            high = mid
            
    final_c = (low + high) / 2
    print(f"====== BUSCA CONCLUÍDA ======")
    print(f"O valor aproximado para o C ideal é: {final_c:.4f}")
    
    return final_c

A função grid_search_c atua como um método de varrimento exploratório para o espaço de hiperparâmetros. Ela recebe uma lista predefinida de valores e testa os vizinhos adjacentes. Esta estratégia serve para desenhar um mapeamento global do desempenho estatístico e contornar o risco de focar uma otimização num mínimo local inadequado, extraindo as fronteiras da zona de maior eficiência.

In [34]:
def grid_search_c(candidates, num_games_per_side=30, iterations=2000):
    """
    Testa todos os candidatos entre si (par a par adjacente) para identificar
    em que intervalo está o pico real de performance.
    Retorna o intervalo (low, high) com maior potencial.
    """
    print(f"====== GRID SEARCH EM {len(candidates)} PONTOS ======\n")
    scores = {c: 0 for c in candidates}

    for i in range(len(candidates) - 1):
        c1 = candidates[i]
        c2 = candidates[i + 1]
        
        print(f"Confronto: C={c1:.4f} vs C={c2:.4f}")
        c2_won = evaluate_c_matchup(c1, c2, num_games_per_side=num_games_per_side, iterations=iterations)
        
        if c2_won:
            scores[c2] += 1
            print(f"  → Vencedor: C={c2:.4f}\n")
        else:
            scores[c1] += 1
            print(f"  → Vencedor: C={c1:.4f}\n")

    print("Scores finais do grid search:")
    for c, s in sorted(scores.items()):
        print(f"  C={c:.4f} → {s} vitórias")

    best = max(scores, key=scores.get)
    idx = candidates.index(best)
    low  = candidates[max(0, idx - 1)]
    high = candidates[min(len(candidates) - 1, idx + 1)]

    print(f"\n→ Melhor zona identificada: [{low:.4f}, {high:.4f}] (pico em C={best:.4f})\n")
    
    return low, high

Por fim, a função robust_c_search consolida o processo de afinamento numa orquestração única. Inicia-se com a busca em grelha para obter a região global promissora e avança para três instâncias independentes de busca binária. O detalhe fundamental nesta etapa é a introdução de ligeiros desvios (offsets) nos limites de cada busca. O valor resultante é extraído através da mediana estatística das três simulações, conferindo ao modelo MCTS uma tolerância robusta à natureza estocástica dos rollouts.

In [35]:
def robust_c_search(iterations=10000, num_games_grid=30, num_games_binary=50, max_steps=5):

    candidates = [1.0, 1.2, 1.414, 1.6, 1.8, 2.0]
    low, high = grid_search_c(candidates, num_games_per_side=num_games_grid, iterations=iterations)

    if high - low < 0.1:
        margin = 0.1
        low  = max(1.0, low  - margin)
        high = min(2.0, high + margin)
        print(f"  (Intervalo alargado para [{low:.4f}, {high:.4f}] por ser muito estreito)\n")

    print(f"====== BUSCAS BINÁRIAS (3x) NO INTERVALO [{low:.4f}, {high:.4f}] ======\n")
    width = high - low
    offsets = [0, -width * 0.1, width * 0.1]
    results = []

    for i, offset in enumerate(offsets, 1):
        l = max(1.0, low  + offset)
        h = min(2.0, high + offset)
        
        print(f"Busca {i}/3: intervalo [{l:.4f}, {h:.4f}]")
        c = optimize_c_binary_search(low=l, high=h,
                                     max_steps=max_steps,
                                     iterations=iterations,
                                     num_games_per_side=num_games_binary)
        results.append(c)
        print(f"  → Busca {i} convergiu para C={c:.4f}\n")

    sorted_results = sorted(results)
    final_c = sorted_results[len(sorted_results) // 2]

    print(f"====== RESULTADO FINAL ======")
    print(f"  Buscas individuais: {[f'{c:.4f}' for c in results]}")
    print(f"  Mediana (estimativa robusta): C = {final_c:.4f}")

    if max(results) - min(results) > 0.15:
        print(f"  ⚠️  Alta divergência entre buscas ({max(results) - min(results):.4f})")
        print(f"      Considera aumentar num_games_per_side para obter mais precisão.")
    else:
        print(f"  ✅ Buscas convergentes — resultado confiável.")

    return final_c

# Chamada da função de otimização robusta para encontrar o melhor valor de C#
#best_c_value = robust_c_search(iterations=10000, num_games_grid=15, num_games_binary=25, max_steps=4 )
#best_c_value

A análise dos resultados empíricos revela que a margem de vitória entre diferentes parametrizações raramente excede os 10%, evidenciando uma sensibilidade mitigada do modelo a variações finas do hiperparâmetro. A flutuação dos valores ótimos obtidos em múltiplas execuções — com a convergência a alternar entre estimativas díspares, como 1.248 e 1.357 — sugere que a topologia de desempenho em função de $C$ não é estritamente unimodal, apresentando uma multiplicidade de máximos locais que dificultam a identificação de um ótimo global absoluto. Adicionalmente, verificou-se que um volume elevado de simulações por jogada (na ordem das 10.000 iterações) induz uma forte estabilização nos resultados, esbatendo o impacto da constante. Deduz-se, portanto, que a calibração rigorosa de $C$ assume um papel significativamente mais determinante em cenários com restrições computacionais, onde um número reduzido de iterações exige uma otimização precisa do compromisso entre a exploração de novas ramificações e a explotação do conhecimento estatístico já adquirido.

Logo, a equipe decidiu prosseguir com o valor de $C$ de teórico ideal de 1.414.

### Alternatives MCTS

Considerando que o valor de C permanecerá o mesmo e que o valor de iterações foi prefixado em 10000 iterações. Para fins de atingir uma maior precisão de nosso algoritmo algumas estratégias podem ser aplicadas a fim de direcionar o algoritmo a resultados mais constantes, uma vez que as simulações apresentam aleatoriedade.

#### Métodos Heurísticos

Ao observar a execução do MCTS em tempo real, percebeu-se que o algoritmo perdia jogadas que resultariam em uma vitória, além não jogar para que o oponente deixasse de ganhar. Então os dois métodos a seguir foram introduzidos:

*_find_winning_move*: Se houver uma jogada que vença o jogo imediatamente, a IA a faz sem perder tempo calculando a árvore.

*_find_blocking_move*: Se o oponente tiver uma jogada que vença no próximo turno, a IA prioriza o bloqueio.

Esses métodos além de direcionarem a árvore para um estilo de jogo mais eficiente, realiza o chamado de "pruning" antecipado.

In [36]:
def _find_winning_move(board):
    piece = board.current_player
    opponent = 'O' if piece == 'X' else 'X'
    
    for move in board.get_valid_moves():
        b = board.copy()
        b.apply_move(move)
        
        if move[0] == 'pop':
            if b.check_win(piece) and b.check_win(opponent):
                return move
            elif b.check_win(piece):
                return move
        else:
            if b.check_win(piece):
                return move
    return None

def _find_blocking_move(board):
    opponent = 'O' if board.current_player == 'X' else 'X'
    
    b_opp = board.copy()
    b_opp.current_player = opponent 
    
    opp_win_move = _find_winning_move(b_opp)
    
    if opp_win_move and opp_win_move in board.get_valid_moves():
        return opp_win_move
    return None


# Monkey Patching #
Node._find_winning_move = _find_winning_move
Node._find_blocking_move = _find_blocking_move

Como último método heurístico, essa equipa observou que uma das estratégias de jogo para se ter mais vitórias no jogo PopOut era de dominar as colunas centrais, então desenvolveu-se o método *weighted_random_move* o qual atribui pesos maiores a jogadas do tipo 'push' nas colunas conforme se aproximem da coluna central.

Então é adicionado todas as jogadas possíveis a uma lista, sendo as jogadas com peso maior que 1 são adicionadas N vezes, sendo N igual ao seu peso. Por fim, é sorteada a jogada vencedora.

In [37]:
def _weighted_random_move(moves, board):
    center = board.cols // 2
    weights = []
    
    for move_type, col in moves:
        if move_type == 'push':
            w = center - abs(col - center) + 1
        else:
            w = 1
        weights.append(w)

    total = sum(weights)
    r = random.uniform(0, total)
    cumulative = 0
    for move, w in zip(moves, weights):
        cumulative += w
        if r <= cumulative:
            return move
    return moves[-1]

# Monkey Patching #
Node._weighted_random_move = _weighted_random_move

#### Novo método Rollout(Simulação)

A adição de novas heurísticas implica na construção de um novo sistema de Simulação onde tais métodos são aplicados.

Diferente da "Simulação Vanilla", a nova simulação tem priorização em jogadas da seguinte ordem: Vitória -> Bloqueio -> jogar ao Centro ao invés de uma seleção aleatória dos possíveis movimentos a serem feitos.


Constroi-se assim um algoritmo capaz de tomar decisões mais inteligentes porém com custos maiores de processamento devido ao aumento do número de verificações necessárias para serem feitas em cada simulação.

In [38]:
def _rollout(board, ai_piece, max_depth=60):
    sim_board = board.copy()

    for _ in range(max_depth):
        legal_moves = sim_board.get_valid_moves()
        if not legal_moves:
            return 0.5

        win_move = _find_winning_move(sim_board)
        if win_move: return 1.0 if sim_board.current_player == ai_piece else 0.0

        block_move = _find_blocking_move(sim_board)
        if block_move:
            move = block_move
        else:
            move = _weighted_random_move(legal_moves, sim_board)

        current_p = sim_board.current_player
        opponent_p = 'O' if current_p == 'X' else 'X'
        sim_board.apply_move(move)

        if move[0] == 'pop':
            cw = sim_board.check_win(current_p)
            ow = sim_board.check_win(opponent_p)
            if cw and ow: return 1.0 if current_p == ai_piece else 0.0
            elif cw: return 1.0 if current_p == ai_piece else 0.0
            elif ow: return 0.0 if current_p == ai_piece else 1.0
        else:
            if sim_board.check_win(current_p):
                return 1.0 if current_p == ai_piece else 0.0
    return 0.5

# Monkey Patching #
Node._rollout = _rollout

#### MCTS: Heuristico + Movimentos com pesos

O primeiro uso alternativo para o MCTS é chamado de **MCTS: heurístico**, pois no seu proceso de definição de movimento, ele utiliza os novos métodos heurísticos apresentados na nova fase de simulação.

In [39]:
def mcts_best_move(board, iterations=10000, c=1.414):
    ai_piece = board.current_player

    instant_win = _find_winning_move(board)
    if instant_win: return instant_win
        
    instant_block = _find_blocking_move(board)
    if instant_block: return instant_block

    root = Node(board.copy())

    for _ in range(iterations):
        node = root
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child(c)

        if not node.is_terminal() and node.untried_moves:
            move = random.choice(node.untried_moves)
            node.untried_moves.remove(move)
            new_board = node.board.copy()
            new_board.apply_move(move)
            child = Node(new_board, parent=node, move=move)
            node.children.append(child)
            node = child

        result = _rollout(node.board, ai_piece)

        backprop_node = node
        while backprop_node is not None:
            backprop_node.visits += 1
            if backprop_node.player_who_just_moved == ai_piece:
                backprop_node.wins += result
            elif backprop_node.player_who_just_moved is not None:
                backprop_node.wins += (1.0 - result)
            backprop_node = backprop_node.parent

    if not root.children: return board.get_valid_moves()[0]
    return max(root.children, key=lambda n: n.visits).move


#### MCTS: Heuristico + Movimentos com pesos + MultiExpansion

A variante *mcts_multi_expansion_best_move* implementa além das otimizações estratégicas vistas no método anteriormente abordado, um sistema que permite a expansão múltipla de nós filhos. Mas para realizar a etapa 3 de SIMULAÇÃO, ele escolhe um dos nós expandidos de forma aleatória.

A principal inovação reside na fase de Expansão, que deixa de ser unitária para se tornar múltipla. Ao expandir até **n_children** nós em uma única iteração, o algoritmo acelera a ramificação da árvore, permitindo que o cálculo do UCB1 tenha uma base comparativa mais ampla em menos tempo. Somado a isso, a fase de Simulação abandona o comportamento puramente aleatório em favor de um rollout heurístico, que simula jogadores mais 'atentos' às regras básicas de vitória e bloqueio. O resultado final é uma árvore que, embora utilize o mesmo número de iterações, possui dados mais densos e de maior qualidade técnica para a tomada de decisão.

In [40]:
def mcts_multi_expansion_best_move(board, iterations=10000, c=1.414, n_children=3):
    ai_piece = board.current_player
    
    instant_win = _find_winning_move(board)
    if instant_win: return instant_win
        
    instant_block = _find_blocking_move(board)
    if instant_block: return instant_block

    root = Node(board.copy())

    for _ in range(iterations):
        node = root
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child(c)

        expanded_nodes = []
        if not node.is_terminal() and node.untried_moves:
            num_to_expand = min(n_children, len(node.untried_moves))
            for _ in range(num_to_expand):
                move = random.choice(node.untried_moves)
                node.untried_moves.remove(move)
                new_board = node.board.copy()
                new_board.apply_move(move)
                child = Node(new_board, parent=node, move=move)
                node.children.append(child)
                expanded_nodes.append(child)
            
            node = random.choice(expanded_nodes)

        result = _rollout(node.board, ai_piece)

        backprop_node = node
        while backprop_node is not None:
            backprop_node.visits += 1
            if backprop_node.player_who_just_moved == ai_piece:
                backprop_node.wins += result
            elif backprop_node.player_who_just_moved is not None:
                backprop_node.wins += (1.0 - result)
            backprop_node = backprop_node.parent

    if not root.children: return board.get_valid_moves()[0]
    return max(root.children, key=lambda n: n.visits).move